# Digit Recognizer — CNN avec Keras
Compétition Kaggle MNIST : classification de chiffres manuscrits (0–9).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split

print('TensorFlow version:', tf.__version__)

## 1. Chargement des données

In [ ]:
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
train_df = pd.read_csv('/kaggle/input/competitions/digit-recognizer/train.csv')
test_df  = pd.read_csv('/kaggle/input/competitions/digit-recognizer/test.csv')

print('Train shape:', train_df.shape)
print('Test shape :', test_df.shape)
train_df.head()

## 2. Distribution des classes

In [ ]:
train_df['label'].value_counts().sort_index().plot(kind='bar', figsize=(8, 4), title='Distribution des chiffres')
plt.xlabel('Chiffre')
plt.ylabel('Nombre d\'exemples')
plt.tight_layout()
plt.show()

## 3. Pré-traitement

In [ ]:
# Séparation features / labels
y = train_df['label'].values
X = train_df.drop('label', axis=1).values

X_test = test_df.values

# Normalisation [0, 1]
X       = X.astype('float32') / 255.0
X_test  = X_test.astype('float32') / 255.0

# Reshape en images 28×28×1 (grayscale)
X      = X.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

# One-hot encoding des labels
y_cat = keras.utils.to_categorical(y, num_classes=10)

# Split train / validation (90% / 10%)
X_train, X_val, y_train, y_val = train_test_split(
    X, y_cat, test_size=0.1, random_state=42, stratify=y
)

print('X_train:', X_train.shape, '  X_val:', X_val.shape, '  X_test:', X_test.shape)

## 4. Visualisation d'exemples

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(15, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i].reshape(28, 28), cmap='gray')
    ax.set_title(np.argmax(y_train[i]))
    ax.axis('off')
plt.suptitle('Exemples d\'entraînement', y=1.02)
plt.tight_layout()
plt.show()

## 5. Architecture du modèle CNN

In [ ]:
def build_model():
    model = keras.Sequential([
        # Bloc convolutif 1
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(28, 28, 1)),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Bloc convolutif 2
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Bloc convolutif 3
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Dropout(0.25),

        # Fully connected
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')
    ])
    return model

model = build_model()
model.summary()

## 6. Compilation et callbacks

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy', factor=0.5, patience=3,
        min_lr=1e-6, verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=10,
        restore_best_weights=True, verbose=1
    )
]

## 7. Data Augmentation

In [ ]:
datagen = keras.preprocessing.image.ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1
)
datagen.fit(X_train)
print('Data augmentation configurée.')

## 8. Entraînement

In [ ]:
BATCH_SIZE = 64
EPOCHS     = 30

history = model.fit(
    datagen.flow(X_train, y_train, batch_size=BATCH_SIZE),
    epochs=EPOCHS,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    steps_per_epoch=len(X_train) // BATCH_SIZE
)

## 9. Courbes d'apprentissage

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'],     label='Train')
ax1.plot(history.history['val_accuracy'], label='Validation')
ax1.set_title('Accuracy')
ax1.set_xlabel('Époque')
ax1.legend()
ax1.grid(True)

ax2.plot(history.history['loss'],     label='Train')
ax2.plot(history.history['val_loss'], label='Validation')
ax2.set_title('Loss')
ax2.set_xlabel('Époque')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

val_acc = max(history.history['val_accuracy'])
print(f'Meilleure accuracy validation : {val_acc:.4f} ({val_acc*100:.2f}%)')

## 10. Matrice de confusion

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

y_val_pred = np.argmax(model.predict(X_val), axis=1)
y_val_true = np.argmax(y_val, axis=1)

cm = confusion_matrix(y_val_true, y_val_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.xlabel('Prédit')
plt.ylabel('Réel')
plt.title('Matrice de confusion (validation)')
plt.show()

print(classification_report(y_val_true, y_val_pred))

## 11. Prédictions sur le test set et soumission

In [ ]:
predictions = np.argmax(model.predict(X_test), axis=1)

submission = pd.DataFrame({
    'ImageId': np.arange(1, len(predictions) + 1),
    'Label':   predictions
})

submission.to_csv('submission.csv', index=False)
print('submission.csv sauvegardé.')
submission.head(10)

## 12. Sauvegarde du modèle

In [ ]:
model.save('digit_recognizer_cnn.keras')
print('Modèle sauvegardé : digit_recognizer_cnn.keras')